# RankUp pada Semua Dataset

Notebook Kaggle ini membandingkan **MLP**, **TabM**, dan **RealMLP** pada lima dataset ordinal. Setiap kombinasi model–dataset menjalankan empat eksperimen:

1. **Supervised** — hanya `N_LABEL` data berlabel.
2. **Fully-Supervised** — semua label train; upper bound.
3. **RankUp + FESTA** — data tanpa label diberi dua augmentasi range-limited.
4. **RankUp + TabCutMix** — data tanpa label diberi dua augmentasi pertukaran fitur.

Metrik test: **R² ↑, Spearman ↑, MAE ↓, RMSE ↓, QWK ↑**. Split tetap sama; setiap eksperimen dijalankan dengan lima seed dan dilaporkan sebagai mean ± std.


## 1. Setup

In [ ]:
%pip install -q tabm

import gc
import random
import time
from itertools import cycle, islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import (
    cohen_kappa_score, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tabm import TabM
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid", context="notebook")

SEEDS = [42, 123, 456, 789, 101112]
SEED = SEEDS[0]
N_LABEL = 200
EPOCHS = 120
BATCH_SIZE = 32             
SUPERVISED_BATCH_SIZE = 256
MAX_RANKUP_STEPS = 100      
LOG_EVERY = 10
W_RANK = 0.2
W_UNLABELED = 0.2
P_CUTOFF = 0.95
FESTA_BINS = 10
TSNE_MAX_SAMPLES = 500

DATASET_FILES = {
    "Melbourne Airbnb": "oc10_melbourneAirbnb.csv",
    "Student Performance": "oc08_studentPerformance.csv",
    "NHANES": "oc05_nhanes.csv",
    "ESL": "oc06_esl.csv",
    "LEV": "oc05_lev.csv",
}
MODELS = ["MLP", "TabM", "RealMLP"]
EXPERIMENTS = ["Supervised", "Fully-Supervised", "RankUp + FESTA", "RankUp + TabCutMix"]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR = WORK_DIR / "rankup_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
HARDWARE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | jumlah GPU tersedia: {torch.cuda.device_count()}")

## 2. Memuat seluruh dataset

Pencarian file mendukung struktur lokal `dataset/` dan folder Kaggle `/kaggle/input/`. Semua fitur harus numerik dan target harus bernama `y` dengan kelas ordinal berurutan mulai dari 0.

In [ ]:
def find_dataset(filename):
    for root in [Path("dataset"), Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            match = next(root.rglob(filename), None)
            if match is not None:
                return match
    raise FileNotFoundError(f"{filename} tidak ditemukan")


datasets = {}
for name, filename in DATASET_FILES.items():
    path = find_dataset(filename)
    df = pd.read_csv(path)
    if "y" not in df.columns or df.isna().any().any():
        raise ValueError(f"{name}: target 'y' wajib ada dan data tidak boleh missing")
    if len(df.select_dtypes(include=np.number).columns) != len(df.columns):
        raise ValueError(f"{name}: semua kolom harus numerik")
    classes = np.sort(df["y"].unique())
    if not np.array_equal(classes, np.arange(len(classes))):
        raise ValueError(f"{name}: kelas harus 0..K-1, ditemukan {classes.tolist()}")
    datasets[name] = df
    print(f"{name:20} | {df.shape[0]:6,} baris | {df.shape[1] - 1:2} fitur | {len(classes)} kelas")

## 3. Split dan preprocessing

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def prepare_data(df):
    X = df.drop(columns="y").to_numpy(dtype=np.float32)
    y = df["y"].to_numpy(dtype=np.float32)
    indices = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=SEED, stratify=y
    )
    if N_LABEL >= len(train_idx):
        raise ValueError(f"N_LABEL={N_LABEL} harus lebih kecil dari train={len(train_idx)}")
    labeled_idx, unlabeled_idx = train_test_split(
        train_idx, train_size=N_LABEL, random_state=SEED, stratify=y[train_idx]
    )

    x_scaler = StandardScaler().fit(X[train_idx])
    y_scaler = StandardScaler().fit(y[labeled_idx, None])
    X_scaled = x_scaler.transform(X).astype(np.float32)
    y_scaled = y_scaler.transform(y[:, None]).astype(np.float32)

    tensor = lambda value: torch.from_numpy(value)
    return {
        "x_labeled": tensor(X_scaled[labeled_idx]),
        "y_labeled": tensor(y_scaled[labeled_idx]),
        "x_unlabeled": tensor(X_scaled[unlabeled_idx]),
        "x_train": tensor(X_scaled[train_idx]),
        "y_train": tensor(y_scaled[train_idx]),
        "x_test": tensor(X_scaled[test_idx]),
        "y_test": y[test_idx].astype(int),
        "y_scaler": y_scaler,
        "n_features": X.shape[1],
        "class_min": int(y.min()),
        "class_max": int(y.max()),
    }


prepared = {name: prepare_data(df) for name, df in datasets.items()}
for name, data in prepared.items():
    print(f"{name:20} | labeled={len(data['x_labeled'])} | unlabeled={len(data['x_unlabeled'])} | test={len(data['x_test'])}")

## 4. Model

Ketiga model mempunyai satu output regresi dan dua logit ranking. Pada TabM, `k` anggota ensemble dilatih independen sesuai dokumentasi resmi; rata-rata hanya dipakai saat inference.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        self.regressor = nn.Linear(32, 1)
        self.ranker = nn.Linear(32, 2)

    def forward(self, x):
        z = self.encode(x)
        return self.regressor(z), self.ranker(z)

    def encode(self, x):
        return self.encoder(x)


class NTPLinear(nn.Module):
    def __init__(self, in_features, out_features, zero_init=False):
        super().__init__()
        factor = 0.0 if zero_init else 1.0
        self.weight = nn.Parameter(factor * torch.randn(in_features, out_features))
        self.bias = nn.Parameter(factor * torch.randn(1, out_features))
        self.scale = in_features ** -0.5

    def forward(self, x):
        return self.scale * (x @ self.weight) + self.bias


class RealMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_features))
        layers = []
        width = n_features
        for _ in range(3):
            layers += [NTPLinear(width, 256), nn.Mish()]
            width = 256
        self.encoder = nn.Sequential(*layers)
        self.regressor = NTPLinear(width, 1, zero_init=True)
        self.ranker = NTPLinear(width, 2, zero_init=True)

    def forward(self, x):
        z = self.encode(x)
        return self.regressor(z), self.ranker(z)

    def encode(self, x):
        return self.encoder(x * self.scale)


class TabMRankUp(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.model = TabM.make(n_num_features=n_features, d_out=3)

    def forward(self, x):
        output = self.model(x)             # (batch, k, 3)
        return output[..., :1], output[..., 1:]

    def encode(self, x):
        # Semua fitur numerik, jadi input dapat langsung masuk ensemble backbone.
        z = self.model.backbone(self.model.ensemble_view(x))
        return z.mean(dim=1)






## 5. Augmentasi unlabeled

FESTA mengambil nilai acak di dalam bin kuantil fitur yang sama. TabCutMix menukar subset fitur dengan baris donor acak. Probabilitas kecil membentuk weak view, probabilitas lebih besar membentuk strong view.

In [ ]:
def festa_edges(x_train, n_bins=FESTA_BINS):
    quantiles = torch.linspace(0, 1, n_bins + 1)
    return torch.stack([torch.quantile(x_train[:, j], quantiles) for j in range(x_train.shape[1])])


def festa_view(x, edges, probability):
    bin_id = torch.stack([
        torch.bucketize(x[:, j].contiguous(), edges[j, 1:-1].contiguous())
        for j in range(x.shape[1])
    ], dim=1)
    low = edges.gather(1, bin_id.T).T
    high = edges.gather(1, (bin_id + 1).T).T
    sampled = low + torch.rand_like(x) * (high - low)
    return torch.where(torch.rand_like(x) < probability, sampled, x)


def tabcutmix_view(x, probability):
    donor = x[torch.randperm(len(x), device=x.device)]
    return torch.where(torch.rand_like(x) < probability, donor, x)




## 6. Loss RankUp dan evaluasi

Regression loss memakai MAE. Ranking target pasangan adalah 1 jika target pertama lebih besar, 0 jika lebih kecil, dan 0.5 jika sama. QWK memakai prediksi kontinu yang dibulatkan dan di-clip ke rentang kelas; metrik lain tetap memakai skor kontinu.

In [ ]:
def regression_loss(prediction, target):
    if prediction.ndim == 3:
        target = target[:, None, :].expand_as(prediction)
    return F.l1_loss(prediction, target)


def pairwise_logits(rank_logits):
    if rank_logits.ndim == 2:
        return rank_logits[:, None, :] - rank_logits[None, :, :]
    rank_logits = rank_logits.permute(1, 0, 2)
    return rank_logits[:, :, None, :] - rank_logits[:, None, :, :]


def pairwise_targets(target):
    difference = target[:, None, 0] - target[None, :, 0]
    probability = (difference > 0).float() + 0.5 * (difference == 0).float()
    return torch.stack([1 - probability, probability], dim=-1)


def supervised_rank_loss(rank_logits, target):
    logits = pairwise_logits(rank_logits)
    targets = pairwise_targets(target)
    if logits.ndim == 4:
        targets = targets.unsqueeze(0).expand(logits.shape[0], -1, -1, -1)
    return F.cross_entropy(logits.reshape(-1, 2), targets.reshape(-1, 2))


def pseudo_rank_loss(weak_logits, strong_logits):
    weak_pairs = pairwise_logits(weak_logits)
    weak_probability = F.softmax(weak_pairs, dim=-1)
    if weak_probability.ndim == 4:
        weak_probability = weak_probability.mean(0)
    confidence, pseudo_target = weak_probability.max(dim=-1)

    strong_pairs = pairwise_logits(strong_logits)
    if strong_pairs.ndim == 4:
        pseudo_target = pseudo_target.unsqueeze(0).expand(strong_pairs.shape[0], -1, -1)
        confidence = confidence.unsqueeze(0).expand(strong_pairs.shape[0], -1, -1)
    loss = F.cross_entropy(
        strong_pairs.reshape(-1, 2), pseudo_target.reshape(-1), reduction="none"
    )
    mask = confidence.reshape(-1) >= P_CUTOFF
    return (loss[mask].mean() if mask.any() else loss.sum() * 0), mask.float().mean().item()


def evaluate(y_true, score, class_min, class_max):
    prediction = np.clip(np.rint(score), class_min, class_max).astype(int)
    spearman = spearmanr(y_true, score).statistic
    return {
        "R2": r2_score(y_true, score),
        "Spearman": 0.0 if np.isnan(spearman) else spearman,
        "MAE": mean_absolute_error(y_true, score),
        "RMSE": mean_squared_error(y_true, score) ** 0.5,
        "QWK": cohen_kappa_score(y_true, prediction, weights="quadratic"),
    }


# Self-check singkat untuk arah pasangan dan bentuk augmentasi.
demo_x = torch.tensor([[0.0, 1.0], [1.0, 0.0], [2.0, 2.0]])
demo_y = torch.tensor([[0.0], [1.0], [2.0]])
assert pairwise_targets(demo_y)[2, 0, 1] == 1
assert festa_view(demo_x, festa_edges(demo_x, 2), 0.5).shape == demo_x.shape
assert tabcutmix_view(demo_x, 0.5).shape == demo_x.shape
print("Self-check RankUp dan augmentasi: OK")

## 7. DataLoader dan inference

Fungsi kecil ini hanya menyiapkan batch, prediksi test, dan hidden feature untuk t-SNE. Training loop tetap ditulis langsung di setiap bagian eksperimen.

In [ ]:
def make_loader(x, y=None, batch_size=BATCH_SIZE, shuffle=True, seed=SEED):
    dataset = TensorDataset(x) if y is None else TensorDataset(x, y)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed), pin_memory=torch.cuda.is_available(),
    )


def predict(model, data):
    loader = make_loader(data["x_test"], batch_size=1024, shuffle=False)
    predictions = []
    model.eval()
    with torch.inference_mode():
        for (x,) in loader:
            prediction, _ = model(x.to(DEVICE, non_blocking=True))
            if prediction.ndim == 3:
                prediction = prediction.mean(dim=1)
            predictions.append(prediction.cpu().numpy())
    scaled = np.concatenate(predictions).reshape(-1, 1)
    return data["y_scaler"].inverse_transform(scaled).ravel()


def extract_features(model, data):
    loader = make_loader(data["x_test"], batch_size=1024, shuffle=False)
    features = []
    model.eval()
    with torch.inference_mode():
        for (x,) in loader:
            features.append(model.encode(x.to(DEVICE, non_blocking=True)).cpu().numpy())
    return np.concatenate(features)

## 8. Menyiapkan penyimpanan hasil

List berikut menampung metrik, prediksi, history, dan representasi fitur dari semua cell eksperimen.

In [ ]:
results, predictions, histories = [], [], []
representations = {}
results_df = predictions_df = history_df = summary_df = history_summary = pd.DataFrame()

## 9. Helper visualisasi

Setiap fungsi membuat satu jenis visualisasi seperti notebook referensi. Fungsi yang sama dipakai oleh ketiga model agar warna, skala, judul, dan file output konsisten.

In [ ]:
PALETTE = dict(zip(EXPERIMENTS, sns.color_palette("colorblind", len(EXPERIMENTS))))
METRICS = ["R2", "Spearman", "MAE", "RMSE", "QWK"]


def summarize_results(frame):
    aggregations = {
        **{f"{metric}_mean": (metric, "mean") for metric in METRICS},
        **{f"{metric}_std": (metric, "std") for metric in METRICS},
        "training_seconds": ("training_seconds", "mean"),
        "inference_seconds": ("inference_seconds", "mean"),
    }
    return frame.groupby(["dataset", "model", "experiment"], as_index=False).agg(**aggregations)


def get_summary():
    return summarize_results(results_df)


def get_history_summary():
    return history_df.groupby(["dataset", "model", "experiment", "epoch"], as_index=False).mean(numeric_only=True)


def show_model_summary(model_name):
    table = get_summary().query("model == @model_name").copy()
    for metric in METRICS:
        table[metric] = table.apply(lambda row: f"{row[f'{metric}_mean']:.4f} ± {row[f'{metric}_std']:.4f}", axis=1)
    columns = ["dataset", "experiment", *METRICS, "training_seconds", "inference_seconds"]
    display(table[columns].reset_index(drop=True).style.format({"training_seconds": "{:.1f}", "inference_seconds": "{:.3f}"}))


def plot_metrics(model_name):
    for dataset_name in DATASET_FILES:
        subset = get_summary().query("model == @model_name and dataset == @dataset_name")
        figure, axes = plt.subplots(1, 5, figsize=(24, 4.5))
        for axis, metric in zip(axes, METRICS):
            sns.barplot(data=subset, x="experiment", y=f"{metric}_mean", hue="experiment", order=EXPERIMENTS, hue_order=EXPERIMENTS, palette=PALETTE, legend=False, ax=axis)
            axis.set_title(metric + (" ↓" if metric in ["MAE", "RMSE"] else " ↑"))
            axis.set_xlabel(""); axis.tick_params(axis="x", rotation=25)
        figure.suptitle(f"{model_name} — {dataset_name} (mean ± std)", y=1.04, fontsize=15)
        figure.tight_layout(); filename = DATASET_FILES[dataset_name][:-4]
        figure.savefig(OUTPUT_DIR / f"metrics_{model_name.lower()}_{filename}.png", dpi=150, bbox_inches="tight"); plt.show()


def plot_training(model_name):
    for dataset_name in DATASET_FILES:
        subset = get_history_summary().query("model == @model_name and dataset == @dataset_name")
        rankup = subset[subset["experiment"].str.startswith("RankUp")]
        rank_losses = rankup.melt(id_vars=["epoch", "experiment"], value_vars=["rank_labeled", "rank_unlabeled"], var_name="component", value_name="value")
        figure, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        sns.lineplot(data=subset, x="epoch", y="loss", hue="experiment", palette=PALETTE, ax=axes[0])
        sns.lineplot(data=rank_losses, x="epoch", y="value", hue="experiment", style="component", palette=PALETTE, ax=axes[1])
        sns.lineplot(data=rankup, x="epoch", y="accepted", hue="experiment", palette=PALETTE, ax=axes[2])
        axes[0].set_title("Total training loss"); axes[1].set_title("Breakdown ranking loss"); axes[2].set_title("Pasangan pseudo-label diterima"); axes[2].set_ylim(0, 1)
        figure.suptitle(f"{model_name} — {dataset_name} (mean antar-seed)", y=1.04, fontsize=15); figure.tight_layout()
        filename = DATASET_FILES[dataset_name][:-4]; figure.savefig(OUTPUT_DIR / f"loss_{model_name.lower()}_{filename}.png", dpi=150, bbox_inches="tight"); plt.show()


def plot_predictions(model_name):
    for dataset_name in DATASET_FILES:
        subset = predictions_df.query("model == @model_name and dataset == @dataset_name").groupby(["experiment", "row", "y_true"], as_index=False)["y_score"].mean()
        subset["residual"] = subset["y_score"] - subset["y_true"]
        low, high = subset[["y_true", "y_score"]].min().min(), subset[["y_true", "y_score"]].max().max()
        figure, axes = plt.subplots(1, 2, figsize=(13, 5))
        sns.scatterplot(data=subset, x="y_true", y="y_score", hue="experiment", palette=PALETTE, alpha=0.45, ax=axes[0]); axes[0].plot([low, high], [low, high], "--", color="black", linewidth=1); axes[0].set_title("Prediksi vs target (mean seed)")
        sns.histplot(data=subset, x="residual", hue="experiment", palette=PALETTE, kde=True, bins=20, alpha=0.35, ax=axes[1]); axes[1].axvline(0, linestyle="--", color="black", linewidth=1); axes[1].set_title("Distribusi residual")
        figure.suptitle(f"{model_name} — {dataset_name}", y=1.03, fontsize=15); figure.tight_layout(); filename = DATASET_FILES[dataset_name][:-4]
        figure.savefig(OUTPUT_DIR / f"prediction_{model_name.lower()}_{filename}.png", dpi=150, bbox_inches="tight"); plt.show()


def plot_tsne(model_name):
    rng = np.random.default_rng(SEED)
    for dataset_name, data in prepared.items():
        n_samples = min(TSNE_MAX_SAMPLES, len(data["y_test"])); sample_idx = np.sort(rng.choice(len(data["y_test"]), n_samples, replace=False)); targets = data["y_test"][sample_idx]
        figure, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
        for axis, experiment in zip(axes, EXPERIMENTS):
            features = representations[(dataset_name, model_name, experiment, SEEDS[0])][sample_idx]
            embedding = TSNE(n_components=2, perplexity=min(30, n_samples - 1), init="pca", learning_rate="auto", random_state=SEED).fit_transform(features)
            scatter = axis.scatter(embedding[:, 0], embedding[:, 1], c=targets, cmap="viridis", s=18, alpha=0.8); axis.set_title(experiment); axis.set_xlabel("t-SNE 1"); axis.set_ylabel("t-SNE 2")
        figure.colorbar(scatter, ax=axes, label="Target ordinal"); figure.suptitle(f"Representasi fitur {model_name} — {dataset_name} (seed {SEEDS[0]})", y=1.05, fontsize=15)
        filename = DATASET_FILES[dataset_name][:-4]; figure.savefig(OUTPUT_DIR / f"tsne_{model_name.lower()}_{filename}.png", dpi=150, bbox_inches="tight"); plt.show()


# 10. Eksperimen MLP

MLP pembanding memiliki encoder 64 → 32, regression head, dan ranking head.

## 10.1 Supervised MLP

Hanya `N_LABEL` label dari setiap dataset yang dipakai.

In [ ]:
def train_mlp_supervised(data, seed):
    set_seed(seed)
    model = MLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = F.l1_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_mlp_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "MLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    mlp_supervised = results_df.query("model == 'MLP' and experiment == 'Supervised'")
    mlp_supervised

## 10.2 Fully-Supervised MLP

Semua label train dipakai sebagai upper bound.

In [ ]:
def train_mlp_fully_supervised(data, seed):
    set_seed(seed)
    model = MLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = make_loader(data["x_train"], data["y_train"], SUPERVISED_BATCH_SIZE, seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = F.l1_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Fully-Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_mlp_fully_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "MLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    mlp_fully = results_df.query("model == 'MLP' and experiment == 'Fully-Supervised'")
    mlp_fully

## 10.3 RankUp + FESTA MLP

`N_LABEL` label melatih regresi dan ranking supervised; sisa train dipakai melalui pseudo-ranking dengan range-limited augmentation.

In [ ]:
def train_mlp_rankup_festa(data, seed):
    set_seed(seed)
    model = MLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    edges = festa_edges(data["x_train"]).to(DEVICE)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = festa_view(x_u, edges, 0.1)
            strong = festa_view(x_u, edges, 0.3)

            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = F.l1_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + FESTA"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_mlp_rankup_festa(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "MLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    mlp_festa = results_df.query("model == 'MLP' and experiment == 'RankUp + FESTA'")
    mlp_festa

## 10.4 RankUp + TabCutMix MLP

Struktur RankUp sama, tetapi weak/strong view dibuat dengan pertukaran subset fitur antarbaris unlabeled.

In [ ]:
def train_mlp_rankup_tabcutmix(data, seed):
    set_seed(seed)
    model = MLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = tabcutmix_view(x_u, 0.1)
            strong = tabcutmix_view(x_u, 0.5)

            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = F.l1_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + TabCutMix"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_mlp_rankup_tabcutmix(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "MLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "MLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    mlp_tabcutmix = results_df.query("model == 'MLP' and experiment == 'RankUp + TabCutMix'")
    mlp_tabcutmix

## 10.5 Hasil dan visualisasi MLP

Tabel, metrik, loss, prediksi, residual, dan representasi fitur ditampilkan setelah seluruh eksperimen MLP selesai.

In [ ]:
show_model_summary("MLP")

In [ ]:
plot_metrics("MLP")

In [ ]:
plot_training("MLP")

In [ ]:
plot_predictions("MLP")

In [ ]:
plot_tsne("MLP")

# 11. Eksperimen TabM

TabM memakai ensemble efisien `k` submodel. Loss setiap anggota dihitung independen; prediksi dirata-ratakan saat inference.

## 11.1 Supervised TabM

In [ ]:
def train_tabm_supervised(data, seed):
    set_seed(seed)
    model = TabMRankUp(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
    loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = regression_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nTabM | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_tabm_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "TabM", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    tabm_supervised = results_df.query("model == 'TabM' and experiment == 'Supervised'")
    tabm_supervised

## 11.2 Fully-Supervised TabM

In [ ]:
def train_tabm_fully_supervised(data, seed):
    set_seed(seed)
    model = TabMRankUp(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
    loader = make_loader(data["x_train"], data["y_train"], SUPERVISED_BATCH_SIZE, seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = regression_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Fully-Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nTabM | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_tabm_fully_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "TabM", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    tabm_fully = results_df.query("model == 'TabM' and experiment == 'Fully-Supervised'")
    tabm_fully

## 11.3 RankUp + FESTA TabM

In [ ]:
def train_tabm_rankup_festa(data, seed):
    set_seed(seed)
    model = TabMRankUp(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    edges = festa_edges(data["x_train"]).to(DEVICE)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = festa_view(x_u, edges, 0.1)
            strong = festa_view(x_u, edges, 0.3)
            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = regression_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + FESTA"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nTabM | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_tabm_rankup_festa(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "TabM", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    tabm_festa = results_df.query("model == 'TabM' and experiment == 'RankUp + FESTA'")
    tabm_festa

## 11.4 RankUp + TabCutMix TabM

In [ ]:
def train_tabm_rankup_tabcutmix(data, seed):
    set_seed(seed)
    model = TabMRankUp(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=3e-4)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = tabcutmix_view(x_u, 0.1)
            strong = tabcutmix_view(x_u, 0.5)
            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = regression_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + TabCutMix"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nTabM | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_tabm_rankup_tabcutmix(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "TabM", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "TabM", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    tabm_tabcutmix = results_df.query("model == 'TabM' and experiment == 'RankUp + TabCutMix'")
    tabm_tabcutmix

## 11.5 Hasil dan visualisasi TabM

In [ ]:
show_model_summary("TabM")

In [ ]:
plot_metrics("TabM")

In [ ]:
plot_training("TabM")

In [ ]:
plot_predictions("TabM")

In [ ]:
plot_tsne("TabM")

# 12. Eksperimen RealMLP-RankUp

Backbone mengikuti komponen RealMLP regresi: feature scaling, tiga NTPLinear 256 unit, dan Mish. Dua output head ditambahkan agar backbone yang sama dapat dipakai pada seluruh eksperimen RankUp.

## 12.1 Supervised RealMLP-RankUp

In [ ]:
def train_realmlp_supervised(data, seed):
    set_seed(seed)
    model = RealMLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = F.l1_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nRealMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_realmlp_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "RealMLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    realmlp_supervised = results_df.query("model == 'RealMLP' and experiment == 'Supervised'")
    realmlp_supervised

## 12.2 Fully-Supervised RealMLP-RankUp

In [ ]:
def train_realmlp_fully_supervised(data, seed):
    set_seed(seed)
    model = RealMLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = make_loader(data["x_train"], data["y_train"], SUPERVISED_BATCH_SIZE, seed=seed)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction, _ = model(x)
            loss = F.l1_loss(prediction, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"epoch": epoch, "loss": np.mean(losses)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(losses):.4f}")
    return model, history


experiment = "Fully-Supervised"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nRealMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_realmlp_fully_supervised(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "RealMLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    realmlp_fully = results_df.query("model == 'RealMLP' and experiment == 'Fully-Supervised'")
    realmlp_fully

## 12.3 RankUp + FESTA RealMLP-RankUp

In [ ]:
def train_realmlp_rankup_festa(data, seed):
    set_seed(seed)
    model = RealMLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    edges = festa_edges(data["x_train"]).to(DEVICE)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = festa_view(x_u, edges, 0.1)
            strong = festa_view(x_u, edges, 0.3)
            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = F.l1_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + FESTA"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nRealMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_realmlp_rankup_festa(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "RealMLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    realmlp_festa = results_df.query("model == 'RealMLP' and experiment == 'RankUp + FESTA'")
    realmlp_festa

## 12.4 RankUp + TabCutMix RealMLP-RankUp

In [ ]:
def train_realmlp_rankup_tabcutmix(data, seed):
    set_seed(seed)
    model = RealMLP(data["n_features"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    labeled_loader = make_loader(data["x_labeled"], data["y_labeled"], seed=seed)
    unlabeled_loader = make_loader(data["x_unlabeled"], seed=seed)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        totals, rank_labs, rank_unlabs, accepted = [], [], [], []
        labeled_batches = cycle(labeled_loader)
        for (x_u,) in islice(unlabeled_loader, MAX_RANKUP_STEPS):
            x_l, y_l = next(labeled_batches)
            x_l, y_l, x_u = x_l.to(DEVICE), y_l.to(DEVICE), x_u.to(DEVICE)
            weak = tabcutmix_view(x_u, 0.1)
            strong = tabcutmix_view(x_u, 0.5)
            optimizer.zero_grad(set_to_none=True)
            pred_l, rank_l = model(x_l)
            loss_reg = F.l1_loss(pred_l, y_l)
            loss_rank_lab = supervised_rank_loss(rank_l, y_l)
            with torch.no_grad():
                _, rank_weak = model(weak)
            _, rank_strong = model(strong)
            loss_rank_unlab, accept_rate = pseudo_rank_loss(rank_weak, rank_strong)
            loss = loss_reg + W_RANK * loss_rank_lab + W_UNLABELED * loss_rank_unlab
            loss.backward()
            optimizer.step()
            totals.append(loss.item())
            rank_labs.append(loss_rank_lab.item())
            rank_unlabs.append(loss_rank_unlab.item())
            accepted.append(accept_rate)
        history.append({"epoch": epoch, "loss": np.mean(totals), "rank_labeled": np.mean(rank_labs), "rank_unlabeled": np.mean(rank_unlabs), "accepted": np.mean(accepted)})
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == EPOCHS:
            print(f"  epoch {epoch:03d}/{EPOCHS} | loss={np.mean(totals):.4f} | accepted={np.mean(accepted):.1%}")
    return model, history


experiment = "RankUp + TabCutMix"
for seed in SEEDS:
    print(f"\nSeed {seed}")
    for dataset_name, data in prepared.items():
        print(f"\nRealMLP | {experiment} | {dataset_name}")
        started = time.perf_counter()
        model, history = train_realmlp_rankup_tabcutmix(data, seed)
        training_seconds = time.perf_counter() - started
        inference_started = time.perf_counter()
        score = predict(model, data)
        inference_seconds = time.perf_counter() - inference_started
        metrics = evaluate(data["y_test"], score, data["class_min"], data["class_max"])
        representations[(dataset_name, "RealMLP", experiment, seed)] = extract_features(model, data)
        results.append({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "hardware": HARDWARE, "training_seconds": training_seconds, "inference_seconds": inference_seconds, **metrics})
        predictions.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, "row": i, "y_true": true, "y_score": pred, "residual": pred - true} for i, (true, pred) in enumerate(zip(data["y_test"], score)))
        histories.extend({"dataset": dataset_name, "model": "RealMLP", "experiment": experiment, "seed": seed, **row} for row in history)
        print(pd.Series(metrics).round(4).to_dict())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    predictions_df = pd.DataFrame(predictions)
    history_df = pd.DataFrame(histories)
    results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
    realmlp_tabcutmix = results_df.query("model == 'RealMLP' and experiment == 'RankUp + TabCutMix'")
    realmlp_tabcutmix

## 12.5 Hasil dan visualisasi RealMLP-RankUp

In [ ]:
show_model_summary("RealMLP")

In [ ]:
plot_metrics("RealMLP")

In [ ]:
plot_training("RealMLP")

In [ ]:
plot_predictions("RealMLP")

In [ ]:
plot_tsne("RealMLP")

# 13. Perbandingan akhir seluruh model

In [ ]:
# Ringkasan lima seed untuk tabel dan visualisasi
results_df = pd.DataFrame(results)
predictions_df = pd.DataFrame(predictions)
history_df = pd.DataFrame(histories)
required_result_columns = {"dataset", "model", "experiment", *METRICS}
missing_result_columns = required_result_columns - set(results_df.columns)
if missing_result_columns:
    raise ValueError(f"Kolom hasil belum lengkap: {sorted(missing_result_columns)}")
expected_result_rows = len(DATASET_FILES) * len(MODELS) * len(EXPERIMENTS) * len(SEEDS)
if len(results_df) != expected_result_rows:
    raise ValueError(f"Jumlah hasil {len(results_df)} tidak sesuai ekspektasi {expected_result_rows}. Ada eksperimen yang belum selesai.")
for frame_name, frame, required_columns in [
    ("predictions", predictions_df, {"dataset", "model", "experiment", "y_true", "y_score", "residual"}),
    ("history", history_df, {"dataset", "model", "experiment", "epoch", "loss"}),
]:
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Kolom {frame_name} belum lengkap: {sorted(missing_columns)}")
summary_df = summarize_results(results_df)
history_summary = history_df.groupby(["dataset", "model", "experiment", "epoch"], as_index=False).mean(numeric_only=True)
results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "metrics_summary_mean_std.csv", index=False)
summary_df.head()


In [ ]:
for dataset_name in DATASET_FILES:
    table = summary_df[summary_df["dataset"] == dataset_name].copy()
    for metric in METRICS:
        table[metric] = table.apply(lambda row: f"{row[f'{metric}_mean']:.4f} ± {row[f'{metric}_std']:.4f}", axis=1)
    columns = ["model", "experiment", *METRICS, "training_seconds", "inference_seconds"]
    display(table[columns].reset_index(drop=True).style.format({"training_seconds": "{:.1f}", "inference_seconds": "{:.3f}"}).set_caption(dataset_name))


## 13.1 Perbandingan metrik

Kelima metrik mendapat subplot sendiri karena skala dan arah optimumnya berbeda. Layout 2×3 memberi ruang untuk label model dan satu legenda bersama.

In [ ]:
metric_titles = {
    "R2": "R² ↑", "Spearman": "Spearman ↑", "MAE": "MAE ↓",
    "RMSE": "RMSE ↓", "QWK": "QWK ↑",
}

for dataset_name in DATASET_FILES:
    subset = summary_df[summary_df["dataset"] == dataset_name]
    figure, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()

    for axis, metric in zip(axes, METRICS):
        sns.barplot(
            data=subset, x="model", y=f"{metric}_mean", hue="experiment",
            order=MODELS, hue_order=EXPERIMENTS, palette=PALETTE, ax=axis,
        )
        axis.set_title(metric_titles[metric], fontweight="bold")
        axis.set_xlabel("")
        axis.set_ylabel(metric)
        axis.grid(axis="y", alpha=0.25)
        legend = axis.get_legend()
        if legend is not None:
            legend.remove()

    axes[-1].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles, labels, title="Eksperimen", loc="lower center",
        bbox_to_anchor=(0.5, 0.01), ncol=4, frameon=False,
    )
    figure.suptitle(f"Perbandingan semua model — {dataset_name}", fontsize=16, fontweight="bold")
    figure.tight_layout(rect=(0, 0.08, 1, 0.95))
    filename = DATASET_FILES[dataset_name][:-4]
    figure.savefig(OUTPUT_DIR / f"comparison_metrics_{filename}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 13.2 Perbandingan kurva training

Kurva loss membantu mendeteksi training yang belum konvergen atau tidak stabil. Loss hanya boleh dibandingkan dalam model yang sama.

In [ ]:
for dataset_name in DATASET_FILES:
    subset = history_df[history_df["dataset"] == dataset_name]
    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
    for axis, model_name in zip(axes, MODELS):
        sns.lineplot(
            data=subset[subset["model"] == model_name], x="epoch", y="loss",
            hue="experiment", hue_order=EXPERIMENTS, palette=PALETTE, ax=axis,
        )
        axis.set_title(model_name)
        axis.set_ylabel("Training loss")
        if axis is not axes[0]:
            legend = axis.get_legend()
            if legend is not None:
                legend.remove()
    axes[0].legend(title="Eksperimen", bbox_to_anchor=(0, 1.25), loc="upper left", ncol=2)
    figure.suptitle(dataset_name, y=1.08, fontsize=15)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"loss_{DATASET_FILES[dataset_name][:-4]}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 13.3 Perbandingan prediksi dan residual

Scatter menunjukkan kedekatan skor kontinu dengan garis ideal. Distribusi residual menunjukkan bias dan sebaran error.

In [ ]:
for dataset_name in DATASET_FILES:
    subset = predictions_df[predictions_df["dataset"] == dataset_name]
    grid = sns.relplot(
        data=subset, x="y_true", y="y_score", row="model", col="experiment",
        row_order=MODELS, col_order=EXPERIMENTS, kind="scatter", alpha=0.35, palette=PALETTE,
        height=3, aspect=1, facet_kws={"sharex": True, "sharey": True},
    )
    low, high = subset[["y_true", "y_score"]].min().min(), subset[["y_true", "y_score"]].max().max()
    for axis in grid.axes.flat:
        axis.plot([low, high], [low, high], "--", color="black", linewidth=1)
    grid.figure.suptitle(f"Prediksi vs target — {dataset_name}", y=1.02)
    grid.figure.savefig(OUTPUT_DIR / f"prediction_{DATASET_FILES[dataset_name][:-4]}.png", dpi=150, bbox_inches="tight")
    plt.show()

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    for axis, model_name in zip(axes, MODELS):
        sns.kdeplot(
            data=subset[subset["model"] == model_name], x="residual", hue="experiment",
            hue_order=EXPERIMENTS, palette=PALETTE, common_norm=False, ax=axis,
        )
        axis.axvline(0, linestyle="--", color="black", linewidth=1)
        axis.set_title(model_name)
    figure.suptitle(f"Distribusi residual — {dataset_name}", y=1.02)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"residual_{DATASET_FILES[dataset_name][:-4]}.png", dpi=150, bbox_inches="tight")
    plt.show()

# 14. File output

Setelah **Save & Run All**, unduh folder `rankup_results` dari tab Output Kaggle. Isinya:

- `metrics.csv`: lima metrik, hardware, waktu training/inference, dan seed tiap run.
- `metrics_summary_mean_std.csv`: ringkasan mean ± std dari lima seed.
- `predictions.csv`: target, skor kontinu, residual, dan seed setiap sampel test.
- `history.csv`: loss per epoch dan seed.
- PNG metrik, loss, prediksi, residual, dan t-SNE.

Pada TabM, representasi t-SNE memakai seed pertama agar visualisasi tetap sederhana.
